<img src="../../../assets/images/logos/ucu_logo_clean.svg" alt="UCU Logo" width="200" style="float: right; margin: 0 0 10px 10px;"/>

### Matemáticas para Aprendizaje Automático - 2026

--------
## Laboratorio 3.2: Funciones Vectoriales, Jacobiano, Matriz Hessiana y Backpropagation

#### Objetivos

- Implementar el Jacobiano de una función vectorial, analíticamente y por diferencias finitas, y aplicar la regla de la cadena a una composición de funciones.
- Calcular la matriz Hessiana de una función escalar, analíticamente y por diferencias centrales, y verificar su simetría.
- Implementar a mano el *forward* y el *backward pass* de una neurona y de una red de dos capas, aplicando la regla de la cadena desde la pérdida hacia los parámetros.
- Contrastar los gradientes propios contra diferencias finitas y contra la diferenciación automática de PyTorch.

## Introducción

Entrenar un modelo de aprendizaje automático es minimizar una función de pérdida respecto de sus parámetros, y casi todos los algoritmos que lo hacen necesitan el **gradiente** de esa pérdida. Un modelo, y en particular una red neuronal, es una composición de funciones sencillas: transformaciones lineales seguidas de no linealidades. Su gradiente se obtiene entonces con la **regla de la cadena** para funciones vectoriales, y el objeto que la hace funcionar es el **Jacobiano**, la matriz de derivadas parciales que generaliza al gradiente cuando la función tiene varias salidas.

En la Parte 1 vas a implementar el Jacobiano y la regla de la cadena sobre un ejemplo chico, y a estimar Jacobianos por diferencias finitas. En la Parte 2 vas a pasar a las derivadas de segundo orden con la matriz Hessiana. En la Parte 3 vas a aplicar la regla de la cadena a una neurona y a una red de dos capas, que es exactamente lo que hace el algoritmo de *backpropagation*, y a comparar el resultado con PyTorch. Cada implementación se valida contra un resultado analítico, contra diferencias finitas o contra una librería de referencia.

In [10]:
%pip install torch


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
# Librerías necesarias
import numpy as np
import scipy.optimize
import torch
import torch.nn as nn

## Parte 1: Jacobiano y Regla de la Cadena

Sea $f: \mathbb{R}^n \to \mathbb{R}^m$ con componentes $f_1, \dots, f_m$. Su **Jacobiano** en $x$ es la matriz $J_f(x) \in \mathbb{R}^{m \times n}$ cuya fila $i$ es el gradiente de $f_i$:

$$J_f(x) = \begin{bmatrix} \dfrac{\partial f_1}{\partial x_1} & \cdots & \dfrac{\partial f_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \dfrac{\partial f_m}{\partial x_1} & \cdots & \dfrac{\partial f_m}{\partial x_n} \end{bmatrix}.$$

Cuando $m = 1$ el Jacobiano es una matriz de $1 \times n$: el **gradiente escrito como fila**, que es la convención del curso.

**Regla de la cadena.** Si $g: \mathbb{R}^n \to \mathbb{R}^k$ y $f: \mathbb{R}^k \to \mathbb{R}^m$ son diferenciables, el Jacobiano de la composición $h = f \circ g$ es el producto de los Jacobianos:

$$J_h(x) = J_f\big(g(x)\big)\, J_g(x).$$

Para una $f$ escalar ($m = 1$) eso se lee $\nabla h(x) = \nabla f\big(g(x)\big)\, J_g(x)$, con $\nabla f$ como fila de $1 \times k$ y $J_g$ de $k \times n$. En NumPy, con arreglos 1D, ese producto es `grad_f @ J_g`.

**Diferencias finitas.** Cada columna del Jacobiano mide cómo cambia $f$ al mover una sola coordenada. Perturbando la coordenada $j$ con el vector canónico $e_j$,

$$J_f(x)\, e_j = \frac{\partial f}{\partial x_j}(x) \approx \frac{f(x + \varepsilon e_j) - f(x)}{\varepsilon}.$$

Esta **diferencia hacia adelante** tiene un error de truncamiento $O(\varepsilon)$, pero el cociente también amplifica el error de redondeo de $f$ en un factor $1/\varepsilon$. El paso $\varepsilon \approx 10^{-8} \approx \sqrt{\epsilon_{\text{máq}}}$ equilibra ambos errores, y por eso las estimaciones coinciden con el valor exacto en unos ocho dígitos, no en dieciséis.

En esta parte vas a aplicar la regla de la cadena a una composición concreta y a implementar el Jacobiano por diferencias finitas. Los resultados se verifican contra el cálculo a mano y contra `scipy.optimize.approx_fprime`.

### Ejercicio L3.2.1: Regla de la Cadena en un Ejemplo

Sean $g: \mathbb{R}^2 \to \mathbb{R}^2$ y $f: \mathbb{R}^2 \to \mathbb{R}$ dadas por

$$g(x, y) = (x y,\; x + y), \qquad f(u, v) = u^2 + v^2,$$

y la composición $h(x, y) = f\big(g(x, y)\big)$, con $h: \mathbb{R}^2 \to \mathbb{R}$.

**a)** Implementá `g`, `f`, el Jacobiano `jacobian_g(x, y)` y el gradiente `grad_f(u, v)`.

**b)** Implementá `grad_h_chain_rule(x, y)`, que calcula $\nabla h(x, y) = \nabla f\big(g(x, y)\big)\, J_g(x, y)$ usando las funciones de **a)**.

**c)** Derivá $h$ a mano, evaluá su gradiente en $(2, 3)$ y guardalo en `grad_analytical`.

**Nota:** el bloque de verificación compara tus resultados contra `scipy.optimize.approx_fprime`, que estima derivadas por diferencias finitas. Si la regla de la cadena falla pero el gradiente analítico pasa, revisá el orden del producto en **b)**.

In [12]:
def g(x, y):
    """
    Función vectorial g: R^2 -> R^2, g(x, y) = (x y, x + y).

    Args:
        x, y: escalares

    Returns:
        array 1D de largo 2 con las componentes (u, v)
    """
    u = x*y
    v = x+y
    return np.array([u, v])


def f(u, v):
    """Función escalar f: R^2 -> R, f(u, v) = u^2 + v^2."""
    return u**2+v**2 #es la formula 


def h(x, y):
    """Función compuesta h = f o g: R^2 -> R."""
    u, v = g(x, y)
    return f(u, v)


def jacobian_g(x, y):
    """
    Jacobiano analítico de g en (x, y).

    Returns:
        J: matriz 2 x 2 con J[i, j] = derivada de la componente i respecto de la variable j
    """
    J = np.array([[y, x],
                 [1, 1]]) #cada fila es una componente de g y las columasn variables de (x,y)
    return J


def grad_f(u, v):
    """
    Gradiente analítico de f en (u, v).

    Returns:
        array 1D de largo 2
    """
    return np.array([2*u, 2*v])


def grad_h_chain_rule(x, y):
    """
    Gradiente de h en (x, y) por la regla de la cadena: grad h = grad f(g(x, y)) J_g(x, y).

    Returns:
        array 1D de largo 2
    """
    grad_h = grad_f(*g(x, y)) @ jacobian_g(x, y) # regla de la cadena: delta f evaluado en g(x,y), por el Jacobiano de g
    return grad_h


x0, y0 = 2.0, 3.0
grad_manual = grad_h_chain_rule(x0, y0)

# c) Gradiente analítico de h en (2, 3), derivado a mano
grad_analytical = np.array([46.0, 34.0])

# ── Verificación ─────────────────────────────────────────────────────────────
# Referencia independiente: diferencias finitas de scipy sobre g y sobre h.
punto = np.array([x0, y0])
J_g_ref = scipy.optimize.approx_fprime(punto, lambda p: g(*p), 1e-7)
grad_h_ref = scipy.optimize.approx_fprime(punto, lambda p: h(*p), 1e-7)

assert np.isclose(h(x0, y0), 61.0), "h(2, 3) = 6^2 + 5^2 = 61: revisá g y f"
assert np.allclose(jacobian_g(x0, y0), J_g_ref, atol=1e-5), "El Jacobiano de g no coincide con la estimación por diferencias finitas"
assert np.allclose(grad_analytical, grad_h_ref, atol=1e-4), "El gradiente analítico no coincide con el de referencia"
assert np.allclose(grad_manual, grad_h_ref, atol=1e-4), "La regla de la cadena no reproduce el gradiente de h"
print("Obtenido (regla de la cadena):", grad_manual)
print("Obtenido (analítico)         :", grad_analytical)
print("Esperado (scipy)             :", grad_h_ref)

Obtenido (regla de la cadena): [46. 34.]
Obtenido (analítico)         : [46. 34.]
Esperado (scipy)             : [46.00000104 34.00000045]


### Ejercicio L3.2.2: Jacobiano por Diferencias Finitas

Implementá una estimación numérica del Jacobiano que sirva para cualquier función $f: \mathbb{R}^n \to \mathbb{R}^m$, y validala en dos casos.

**a)** `numerical_jacobian(f, x, eps)`: devuelve la matriz $m \times n$ cuya columna $j$ es la diferencia hacia adelante $\big(f(x + \varepsilon e_j) - f(x)\big)/\varepsilon$. Si $f$ es escalar, el resultado debe tener forma $1 \times n$.

**b)** Calculá el Jacobiano numérico de $F(x, y, z) = \big(xyz,\; x + y + z,\; x^2 - y^2 + z^2\big)$ en $(1, 2, 3)$. El andamiaje ya trae `vector_function` y su Jacobiano analítico para comparar.

**c)** Calculá el Jacobiano numérico de la función $h$ del ejercicio anterior en $(2, 3)$. Debe coincidir con el gradiente que obtuviste por la regla de la cadena, escrito como fila.

**Nota:** `numerical_jacobian` recibe una función de **un único vector**, mientras que `h` recibe dos escalares. Envolvela con `lambda p: h(*p)`. Para perturbar una coordenada sin modificar `x`, trabajá sobre una copia (`x.copy()`).

In [13]:
def numerical_jacobian(f, x, eps=1e-8):
    """
    Estima el Jacobiano de f: R^n -> R^m por diferencias finitas hacia adelante.

    Args:
        f: función que recibe un array 1D de largo n y devuelve un escalar o un array 1D de largo m
        x: punto de evaluación (array 1D de largo n)
        eps: paso de la diferencia finita

    Returns:
        J: Jacobiano estimado (m x n); si f es escalar, de forma (1 x n)
    """
    x = np.array(x, dtype=float)
    f_x = np.atleast_1d(f(x))
    n = x.shape[0]      # dimensión de entrada
    m = f_x.shape[0]    # dimensión de salida

    J = np.zeros((m, n))
    for j in range(n):
        J[:, j] =(np.atleast_1d(f(x + eps * np.eye(n)[j])) - f_x) / eps  # mueve solo la coordenada j un poquito y mide cuánto cambia f
    return J


def vector_function(x):
    """F(x, y, z) = (xyz, x + y + z, x^2 - y^2 + z^2). Viene dada."""
    x1, x2, x3 = x
    return np.array([
        x1 * x2 * x3,
        x1 + x2 + x3,
        x1**2 - x2**2 + x3**2
    ])


def analytical_jacobian(x):
    """Jacobiano analítico de vector_function. Viene dado."""
    x1, x2, x3 = x
    return np.array([
        [x2 * x3, x1 * x3, x1 * x2],  # derivadas de xyz
        [1, 1, 1],                    # derivadas de x + y + z
        [2 * x1, -2 * x2, 2 * x3]     # derivadas de x^2 - y^2 + z^2
    ])


# b) Jacobiano de F en (1, 2, 3)
x_test = np.array([1.0, 2.0, 3.0])
J_analytical = analytical_jacobian(x_test)
J_numerical = numerical_jacobian(vector_function, x_test)  # Jacobiano numérico de F en (1,2,3)

# c) Jacobiano de h en (2, 3)
J_h_numerical = numerical_jacobian(lambda p: h(*p), np.array([2.0, 3.0]))  # el lambda adapta h para que reciba un solo vector

# ── Verificación ─────────────────────────────────────────────────────────────
assert J_numerical.shape == (3, 3), "El Jacobiano de F: R^3 -> R^3 debe ser de 3 x 3"
assert np.allclose(J_numerical, J_analytical, atol=1e-5), "El Jacobiano numérico de F no coincide con el analítico"
assert J_h_numerical.shape == (1, 2), "Para una función escalar el Jacobiano es 1 x n: el gradiente como fila"
assert np.allclose(J_h_numerical, grad_h_chain_rule(2.0, 3.0).reshape(1, -1), atol=1e-4), \
    "El Jacobiano numérico de h no coincide con la regla de la cadena"
print("Obtenido : J_F numérico =\n", J_numerical)
print("Esperado : J_F analítico =\n", J_analytical)
print(f"Diferencia máxima en J_F: {np.max(np.abs(J_numerical - J_analytical)):.2e}")
print("\nObtenido : J_h numérico =", J_h_numerical)
print("Esperado : regla de la cadena =", grad_h_chain_rule(2.0, 3.0))

Obtenido : J_F numérico =
 [[ 5.99999996  2.99999998  1.99999999]
 [ 0.99999999  0.99999999  0.99999999]
 [ 1.99999999 -3.99999998  5.99999996]]
Esperado : J_F analítico =
 [[ 6.  3.  2.]
 [ 1.  1.  1.]
 [ 2. -4.  6.]]
Diferencia máxima en J_F: 3.65e-08

Obtenido : J_h numérico = [[46.00000025 33.99999997]]
Esperado : regla de la cadena = [46. 34.]


## Parte 2: Matriz Hessiana

Las derivadas parciales de segundo orden surgen al aplicar el operador de derivada parcial dos veces. La **matriz Hessiana** de una función escalar $f: \mathbb{R}^n \to \mathbb{R}$ reúne todas sus derivadas segundas:

$$H_f(x) = \begin{bmatrix} \dfrac{\partial^2 f}{\partial x_1^2} & \cdots & \dfrac{\partial^2 f}{\partial x_1 \partial x_n} \\ \vdots & \ddots & \vdots \\ \dfrac{\partial^2 f}{\partial x_n \partial x_1} & \cdots & \dfrac{\partial^2 f}{\partial x_n^2} \end{bmatrix} \in \mathbb{R}^{n \times n}.$$

Equivalentemente, la Hessiana es el Jacobiano del gradiente. Describe la **curvatura** de $f$: es lo que usan la aproximación de Taylor de segundo orden, la clasificación de puntos críticos y los métodos de optimización de segundo orden.

**Teorema de Clairaut.** Si las derivadas parciales mixtas son continuas, el orden de derivación no importa:

$$\frac{\partial^2 f}{\partial x_i\, \partial x_j} = \frac{\partial^2 f}{\partial x_j\, \partial x_i},$$

y por lo tanto la Hessiana es **simétrica**.

**Diferencias centrales.** Cada entrada de la Hessiana se puede estimar evaluando $f$ en las cuatro esquinas de un pequeño cuadrado alrededor de $x$:

$$\frac{\partial^2 f}{\partial x_i\, \partial x_j}(x) \approx \frac{f(x + \varepsilon e_i + \varepsilon e_j) - f(x + \varepsilon e_i - \varepsilon e_j) - f(x - \varepsilon e_i + \varepsilon e_j) + f(x - \varepsilon e_i - \varepsilon e_j)}{4 \varepsilon^2}.$$

La fórmula es simétrica en $i$ y $j$, tiene error de truncamiento $O(\varepsilon^2)$ y, como divide por $\varepsilon^2$, se usa con un paso más grande que el de la Parte 1 ($\varepsilon \approx 10^{-4}$). En esta parte vas a calcular una Hessiana a mano, estimarla numéricamente y verificar que ambas coinciden y son simétricas.

### Ejercicio L3.2.3: Cálculo de la Matriz Hessiana

Considerá la función $f(x, y, z) = xyz + x^2 + y^2 + z^2$.

**a)** Implementá `f_hess(x)`, que evalúa $f$ en un vector $x = (x, y, z)$.

**b)** Implementá `analytical_hessian(x)`. Derivando dos veces se obtiene

$$H_f(x, y, z) = \begin{bmatrix} 2 & z & y \\ z & 2 & x \\ y & x & 2 \end{bmatrix}.$$

**c)** Implementá `numerical_hessian(f, x, eps)` con la fórmula de diferencias centrales de la introducción y comparala con la Hessiana analítica en $(1, 2, 3)$.

El bloque de verificación agrega por su cuenta un control de simetría sobre la Hessiana numérica.

In [14]:
def f_hess(x):
    """f(x, y, z) = xyz + x^2 + y^2 + z^2."""
    x1, x2, x3 = x
    return x1*x2*x3 + x1**2 + x2**2 + x3**2 #es lo mimso q f(x,y,z) de arriba


def analytical_hessian(x):
    """
    Hessiana analítica de f_hess.

    Returns:
        H: matriz 3 x 3
    """
    x1, x2, x3 = x
    return np.array([[2, x3, x2],
                 [x3, 2, x1],
                 [x2, x1, 2]])  # la matriz de lo de arriba, con x1=x, x2=y, x3=z


def numerical_hessian(f, x, eps=1e-4):
    """
    Estima la Hessiana de f: R^n -> R por diferencias centrales.

    Args:
        f: función escalar que recibe un array 1D de largo n
        x: punto de evaluación (array 1D de largo n)
        eps: paso de la diferencia finita

    Returns:
        H: Hessiana estimada (n x n)
    """
    x = np.array(x, dtype=float)
    D = len(x)
    H = np.zeros((D, D))
    for i in range(D):
        for j in range(D):
            H[i, j] = (f(x + eps*np.eye(D)[i] + eps*np.eye(D)[j]) - f(x + eps*np.eye(D)[i] - eps*np.eye(D)[j]) 
                       - f(x - eps*np.eye(D)[i] + eps*np.eye(D)[j]) + f(x - eps*np.eye(D)[i] - eps*np.eye(D)[j])) / (4*eps**2)  # fórmula de las 4 esquinas
    return H


x_test_hess = np.array([1.0, 2.0, 3.0])
H_ana = analytical_hessian(x_test_hess)
H_num = numerical_hessian(f_hess, x_test_hess)

# ── Verificación ─────────────────────────────────────────────────────────────
# Caso analítico en (1, 2, 3): f = 6 + 1 + 4 + 9 = 20 y H = [[2, z, y], [z, 2, x], [y, x, 2]].
H_esperada = np.array([[2.0, 3.0, 2.0],
                       [3.0, 2.0, 1.0],
                       [2.0, 1.0, 2.0]])

assert np.isclose(f_hess(x_test_hess), 20.0), "f(1, 2, 3) debe valer 20"
assert np.allclose(H_ana, H_esperada), "La Hessiana analítica no coincide con la teórica"
assert np.allclose(H_num, H_ana, atol=1e-4), "La estimación numérica no coincide con la Hessiana analítica"
assert np.allclose(H_num, H_num.T, atol=1e-6), "Por el teorema de Clairaut, la Hessiana debe ser simétrica"
print("Obtenido : H numérica =\n", H_num)
print("Esperado : H analítica =\n", H_ana)
print(f"Diferencia máxima: {np.max(np.abs(H_num - H_ana)):.2e}")

Obtenido : H numérica =
 [[1.99999999 2.99999998 1.99999999]
 [2.99999998 2.00000008 0.99999999]
 [1.99999999 0.99999999 1.99999999]]
Esperado : H analítica =
 [[2. 3. 2.]
 [3. 2. 1.]
 [2. 1. 2.]]
Diferencia máxima: 7.67e-08


## Parte 3: Backpropagation en Redes Neuronales

**Una neurona.** Una neurona con entrada $x \in \mathbb{R}^n$, pesos $w \in \mathbb{R}^n$, sesgo $b \in \mathbb{R}$ y activación sigmoide calcula

$$z = w^\top x + b, \qquad y = \sigma(z) = \frac{1}{1 + e^{-z}}.$$

Con un valor objetivo $t$, la usamos con la pérdida cuadrática $L = \tfrac{1}{2}(y - t)^2$. La sigmoide tiene una derivada que se expresa a partir de sí misma:

$$\sigma'(z) = \sigma(z)\,\big(1 - \sigma(z)\big).$$

**Backpropagation.** La pérdida es la composición $(w, b) \mapsto z \mapsto y \mapsto L$, así que su gradiente sale de la regla de la cadena aplicada **desde la pérdida hacia atrás**:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial y}\,\frac{\partial y}{\partial z}\,\frac{\partial z}{\partial w}, \qquad \frac{\partial L}{\partial b} = \frac{\partial L}{\partial y}\,\frac{\partial y}{\partial z}\,\frac{\partial z}{\partial b}.$$

Los dos primeros factores son comunes, y conviene calcularlos una sola vez y guardarlos. Ésa es la idea central del algoritmo: el *forward pass* evalúa la red y guarda los valores intermedios ($z$, $y$), y el *backward pass* recorre la composición en sentido inverso multiplicando derivadas locales. En una red con varias capas, la derivada respecto de la salida de una capa se reutiliza para obtener las de todas las capas anteriores.

**La red de dos capas.** Con entrada $x \in \mathbb{R}^2$, una capa oculta de dos neuronas y una neurona de salida, ambas con activación sigmoide:

$$z_1 = A_1 x + b_1, \quad a_1 = \sigma(z_1), \quad z_2 = w_2^\top a_1 + b_2, \quad y = \sigma(z_2), \quad L = \tfrac{1}{2}(y - t)^2,$$

con $A_1 \in \mathbb{R}^{2 \times 2}$, $b_1 \in \mathbb{R}^2$, $w_2 \in \mathbb{R}^2$ y $b_2 \in \mathbb{R}$. La sigmoide se aplica elemento a elemento. Es la arquitectura vista en clase y la del problema 7 de la hoja de ejercicios 3.2.

**Diferenciación automática.** Las librerías como PyTorch implementan este mismo algoritmo de forma general: registran las operaciones del *forward pass* y, al llamar a `backward()` sobre la pérdida, aplican la regla de la cadena hacia atrás y dejan el gradiente de cada parámetro en su atributo `.grad`. En esta parte vas a implementar backpropagation a mano, primero para una neurona y después para la red de dos capas, y a verificarlo contra diferencias finitas y contra PyTorch.

### Ejercicio L3.2.4: Gradientes de una Neurona Simple

Completá la clase `SingleNeuron` y verificá sus gradientes con las diferencias finitas de la Parte 1.

**a)** Implementá `sigmoid(z)` y `sigmoid_derivative(z)`.

**b)** Implementá `forward` (que guarda $z$ e $y$ en `self.z` y `self.y`) y `loss`.

**c)** Implementá `backward`, que devuelve $\partial L/\partial w$ (array de largo $n$) y $\partial L/\partial b$ (escalar) siguiendo la cadena $L \to y \to z \to (w, b)$.

**d)** Escribí la pérdida como función de los pesos (con $b$ fijo) y como función del sesgo (con $w$ fijo), y estimá ambos gradientes con `numerical_jacobian`.

**Nota:** la pérdida es escalar, así que `numerical_jacobian` devuelve una matriz de $1 \times n$. Usá `.ravel()` para llevarla a un array 1D.

In [15]:
def sigmoid(z):
    """Función sigmoide, 1 / (1 + exp(-z)), elemento a elemento."""
    return 1 / (1 + np.exp(-z))  # funcion sigmoide


def sigmoid_derivative(z):
    """Derivada de la sigmoide en z, sigmoid(z) (1 - sigmoid(z)), elemento a elemento."""
    return sigmoid(z) * (1 - sigmoid(z))  # la funcion de arriba escrita


class SingleNeuron:
    """
    Neurona con activación sigmoide: y = sigmoid(w^T x + b).
    Tiene un vector de pesos `w` y un sesgo `b`.
    """

    def __init__(self, input_size=2):
        np.random.seed(42)
        self.w = np.random.randn(input_size) * 0.1  # pesos
        self.b = 0.0                                # sesgo

    def set_weights(self, w):
        """Establece los pesos de la neurona."""
        self.w = np.array(w, dtype=float)

    def set_bias(self, b):
        """Establece el sesgo de la neurona."""
        self.b = float(b)

    def forward(self, x):
        """Forward pass: guarda z = w^T x + b e y = sigmoid(z), y devuelve y."""
        self.z = self.w @ x + self.b  # z = wtrasouesta x + b
        self.y = sigmoid(self.z)      # y = sigmoid (z)
        return self.y

    def loss(self, y, target):
        """Pérdida cuadrática L = 1/2 (y - target)^2."""
        return  0.5 * (y - target)**2  

    def backward(self, x, y, target):
        """
        Backward pass. Usa self.z guardado por forward.

        Returns:
            dL_dw: gradiente respecto de los pesos (array 1D de largo n)
            dL_db: gradiente respecto del sesgo (escalar)
        """
        dL_dy = y - target                    # derivada de la loss
        dy_dz = sigmoid_derivative(self.z)    # sigmoide derivado, usando el z guardado en forward
        dL_dz = dL_dy * dy_dz                 # regla de la cadena(se multiplican)
        dL_dw = dL_dz * x                     
        dL_db = dL_dz                         
        return dL_dw, dL_db


# Datos de ejemplo
x = np.array([1.0, 0.5])
w = np.array([0.3, 0.7])
b = 0.2
target = 1.0

neuron = SingleNeuron(input_size=len(x))
neuron.set_weights(w)
neuron.set_bias(b)

y = neuron.forward(x)
loss = neuron.loss(y, target)
dL_dw, dL_db = neuron.backward(x, y, target)


# d) Gradientes numéricos
def loss_as_function_of_w(w_vec):
    """Pérdida de la neurona como función de los pesos, con x, b y target fijos."""
    return 0.5 * (sigmoid(w_vec @ x + b) - target)**2  # misma pérdida, pero variando solo w


def loss_as_function_of_b(b_vec):
    """Pérdida de la neurona como función del sesgo (b_vec: array de largo 1), con x, w y target fijos."""
    return 0.5 * (sigmoid(w @ x + b_vec[0]) - target)**2  # b_vec es un array de largo 1, por eso el [0]


dL_dw_numerical = numerical_jacobian(loss_as_function_of_w, w)[0]  # la única fila de la matriz 1×2
dL_db_numerical = numerical_jacobian(loss_as_function_of_b, [b]).item()  # matriz 1×1 =un numero

# ── Verificación ─────────────────────────────────────────────────────────────
# Caso analítico: z = 0.3 * 1 + 0.7 * 0.5 + 0.2 = 0.85.
y_esperado = 1 / (1 + np.exp(-0.85))

assert np.isclose(sigmoid(0.0), 0.5) and np.isclose(sigmoid_derivative(0.0), 0.25), \
    "sigmoid(0) = 1/2 y sigmoid'(0) = 1/4"
assert np.isclose(y, y_esperado), "La salida de la neurona no coincide con sigmoid(w^T x + b)"
assert np.isclose(loss, 0.5 * (y_esperado - target)**2), "La pérdida no coincide con 1/2 (y - t)^2"
assert np.allclose(dL_dw, dL_dw_numerical, rtol=1e-4, atol=1e-7), "Los gradientes dL/dw no coinciden con los numéricos"
assert np.isclose(dL_db, dL_db_numerical, rtol=1e-4, atol=1e-7), "Los gradientes dL/db no coinciden con los numéricos"
assert np.allclose(dL_dw, dL_db * x), "Como dz/dw = x y dz/db = 1, debe cumplirse dL/dw = (dL/db) x"
print("Obtenido : dL/dw =", dL_dw, "| dL/db =", dL_db)
print("Esperado : dL/dw =", dL_dw_numerical, "| dL/db =", dL_db_numerical, "(diferencias finitas)")

Obtenido : dL/dw = [-0.06281288 -0.03140644] | dL/db = -0.06281287533106923
Esperado : dL/dw = [-0.06281288 -0.03140644] | dL/db = -0.06281287850962158 (diferencias finitas)


### Ejercicio L3.2.5: Comparación con PyTorch

PyTorch calcula gradientes automáticamente. Una neurona como la del ejercicio anterior se arma como una secuencia de una capa lineal seguida de una sigmoide:

```python
neuron_torch = nn.Sequential(
    nn.Linear(2, 1),  # capa lineal: 2 entradas, 1 salida
    nn.Sigmoid()      # activación sigmoide
)
```

`nn.Linear(2, 1)` guarda sus pesos en `neuron_torch[0].weight`, un tensor de forma $1 \times 2$, y su sesgo en `neuron_torch[0].bias`, de forma $1$. El índice `0` se refiere a la primera capa de la secuencia. El andamiaje ya copia en ellos los valores `w` y `b` del ejercicio anterior.

Para calcular gradientes se evalúa la red sobre un tensor de entrada, se calcula la pérdida y se llama a `backward()` **sobre la pérdida**:

```python
salida = neuron_torch(x_torch)
perdida = 0.5 * (salida - target_torch) ** 2
perdida.backward()
neuron_torch[0].weight.grad   # gradiente respecto de los pesos
neuron_torch[0].bias.grad     # gradiente respecto del sesgo
```

**a)** Creá la neurona `neuron_torch` con `nn.Sequential`.

**b)** Creá los tensores de entrada y objetivo con los mismos valores del ejercicio anterior, calculá la salida y la pérdida, y llamá a `backward()`.

**c)** Extraé los gradientes como arrays de NumPy (`.detach().numpy()`) y comparalos con los de tu implementación manual.

**Nota:** PyTorch trabaja por defecto en `float32`, así que los resultados coinciden con los de NumPy en unos siete dígitos, no en dieciséis.

In [16]:
# a) Neurona en PyTorch
neuron_torch = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())  # capa lineal + sigmoide

with torch.no_grad():
    neuron_torch[0].weight.copy_(torch.tensor(w, dtype=torch.float32).unsqueeze(0))
    neuron_torch[0].bias.copy_(torch.tensor([b], dtype=torch.float32))

# b) Forward, pérdida y backward
x_torch = torch.tensor([1.0, 0.5])               # entrada
target_torch = torch.tensor([1.0])               # objetivo
output = neuron_torch(x_torch)                   # forward
loss_torch = 0.5 * (output - target_torch)**2    # L 
loss_torch.backward()                            # los gradientes
# COMPLETAR: llamar a backward() sobre la pérdida

# c) Gradientes como arrays de NumPy
grad_w_torch = neuron_torch[0].weight.grad.numpy()[0]  # 1×2 =array de largo 2
grad_b_torch = neuron_torch[0].bias.grad.item()        # largo 1 =escalar

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.isclose(float(output.detach().numpy().ravel()[0]), y, atol=1e-6), \
    "La salida de PyTorch debe coincidir con la de SingleNeuron"
assert np.allclose(grad_w_torch, dL_dw, atol=1e-6), "Los gradientes dL/dw de PyTorch y de la implementación manual no coinciden"
assert np.isclose(grad_b_torch, dL_db, atol=1e-6), "Los gradientes dL/db de PyTorch y de la implementación manual no coinciden"
print("Obtenido (PyTorch): dL/dw =", grad_w_torch, "| dL/db =", grad_b_torch)
print("Esperado (manual) : dL/dw =", dL_dw, "| dL/db =", dL_db)

Obtenido (PyTorch): dL/dw = [-0.06281286 -0.03140643] | dL/db = -0.06281286478042603
Esperado (manual) : dL/dw = [-0.06281288 -0.03140644] | dL/db = -0.06281287533106923


### Ejercicio L3.2.6: Red de Dos Capas, Implementación Manual frente a PyTorch

Implementá a mano la red de dos capas de la introducción de esta parte y comparala con una red equivalente en PyTorch.

**a)** Completá `ManualTwoLayerNetwork`: el constructor guarda los parámetros, `forward` calcula y guarda $z_1$, $a_1$, $z_2$ e $y$, y `backward` devuelve los gradientes de la pérdida respecto de $A_1$, $b_1$, $w_2$ y $b_2$, en ese orden.

**b)** Creá `pytorch_two_layer_net` con `nn.Sequential`: capa lineal $2 \to 2$, sigmoide, capa lineal $2 \to 1$, sigmoide. El andamiaje copia en ella los mismos pesos y calcula sus gradientes con `backward()`.

**Nota:** reutilizá `sigmoid` y `sigmoid_derivative` del ejercicio L3.2.4. Cada gradiente tiene la misma forma que su parámetro. En particular, $\partial L/\partial A_1$ es una matriz de $2 \times 2$ que se obtiene como un producto externo (`np.outer`) entre la derivada respecto de $z_1$ y la entrada $x$.

In [17]:
class ManualTwoLayerNetwork:
    """
    Red 2 -> 2 -> 1 con activaciones sigmoide:
        z1 = A1 x + b1,  a1 = sigmoid(z1),  z2 = w2^T a1 + b2,  y = sigmoid(z2).
    """

    def __init__(self, A1, b1, w2, b2):
        self.A1 = A1  # guarda los parámetros
        self.b1 = b1
        self.w2 = w2
        self.b2 = b2
    def forward(self, x):
        """Forward pass: guarda los valores intermedios y devuelve la salida y."""
        self.x = x                               # entrada
        self.z1 = self.A1 @ x + self.b1          # z1 = A1·x + b1
        self.a1 = sigmoid(self.z1)               # a1 = sigmpide(z1)
        self.z2 = self.w2 @ self.a1 + self.b2    # z2 = w2traspuesta ·a1 + b2
        self.y = sigmoid(self.z2)                # y = sigmoide(z2)
        return self.y

    def backward(self, x, y, target):
        """
        Backward pass para L = 1/2 (y - target)^2. Usa los valores guardados por forward.

        Returns:
            dL_dA1: gradiente respecto de A1 (2 x 2)
            dL_db1: gradiente respecto de b1 (largo 2)
            dL_dw2: gradiente respecto de w2 (largo 2)
            dL_db2: gradiente respecto de b2 (largo 1)
        """
        dL_dy = y - target                                  # derivada de L
        dL_dz2 = dL_dy * sigmoid_derivative(self.z2)        # pasa por la sigmoide de salida
        dL_dw2 = dL_dz2 * self.a1                           # 
        dL_db2 = dL_dz2                                     # 
        dL_da1 = dL_dz2 * self.w2                           # 
        dL_dz1 = dL_da1 * sigmoid_derivative(self.z1)       # pasa por la sigmoide
        dL_dA1 = np.outer(dL_dz1, x)                        # 2×2: fila i = dL_dz1[i] · x
        dL_db1 = dL_dz1                                     # z1/b1 = 1
        return dL_dA1, dL_db1, dL_dw2, dL_db2


# Parámetros (mismos del problema 7 de la hoja de ejercicios 3.2)
A_1 = np.array([[0.5, 0.2], [0.3, 0.8]])
b_1 = np.array([0.1, 0.4])
w_2 = np.array([0.6, 0.9])
b_2 = np.array([0.2])
manual_two_layer_net = ManualTwoLayerNetwork(A_1, b_1, w_2, b_2)

# b) Red equivalente en PyTorch
pytorch_two_layer_net = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1), nn.Sigmoid())  # 2 2 1

with torch.no_grad():
    pytorch_two_layer_net[0].weight.copy_(torch.tensor(A_1, dtype=torch.float32))
    pytorch_two_layer_net[0].bias.copy_(torch.tensor(b_1, dtype=torch.float32))
    pytorch_two_layer_net[2].weight.copy_(torch.tensor(w_2.reshape(1, -1), dtype=torch.float32))
    pytorch_two_layer_net[2].bias.copy_(torch.tensor(b_2, dtype=torch.float32))

# Datos
x_2c = np.array([1.0, 0.0])
y_target = np.array([0.0])

# Salida y gradientes de la red manual
y_manual = manual_two_layer_net.forward(x_2c)
dL_dA1, dL_db1, dL_dw2, dL_db2 = manual_two_layer_net.backward(x_2c, y_manual, y_target)

# Salida y gradientes de la red PyTorch
x_tensor = torch.tensor(x_2c, dtype=torch.float32)
y_target_tensor = torch.tensor(y_target, dtype=torch.float32)
y_torch = pytorch_two_layer_net(x_tensor)
loss_2c_torch = (0.5 * (y_torch - y_target_tensor) ** 2).sum()
loss_2c_torch.backward()

dL_dA1_torch = pytorch_two_layer_net[0].weight.grad.numpy()
dL_db1_torch = pytorch_two_layer_net[0].bias.grad.numpy()
dL_dw2_torch = pytorch_two_layer_net[2].weight.grad.numpy().ravel()
dL_db2_torch = pytorch_two_layer_net[2].bias.grad.numpy()

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.shape(dL_dA1) == (2, 2), "dL/dA1 debe tener la forma de A1 (2 x 2)"
assert np.allclose(y_manual, y_torch.detach().numpy(), atol=1e-6), "Las salidas de ambas redes no coinciden"
assert np.allclose(dL_dA1, dL_dA1_torch, atol=1e-6), "Los gradientes dL/dA1 no coinciden"
assert np.allclose(dL_db1, dL_db1_torch, atol=1e-6), "Los gradientes dL/db1 no coinciden"
assert np.allclose(dL_dw2, dL_dw2_torch, atol=1e-6), "Los gradientes dL/dw2 no coinciden"
assert np.allclose(dL_db2, dL_db2_torch, atol=1e-6), "Los gradientes dL/db2 no coinciden"
assert np.allclose(dL_dA1[:, 1], 0.0), "Con x = [1, 0], la segunda columna de dL/dA1 debe ser nula"
print("Obtenido (manual) : y =", y_manual, "\n  dL/dA1 =\n", dL_dA1, "\n  dL/db1 =", dL_db1,
      "| dL/dw2 =", dL_dw2, "| dL/db2 =", dL_db2)
print("Esperado (PyTorch): y =", y_torch.detach().numpy(), "\n  dL/dA1 =\n", dL_dA1_torch, "\n  dL/db1 =", dL_db1_torch,
      "| dL/dw2 =", dL_dw2_torch, "| dL/db2 =", dL_db2_torch)

Obtenido (manual) : y = [0.76651972] 
  dL/dA1 =
 [[0.01883104 0.        ]
 [0.0273735  0.        ]] 
  dL/db1 = [0.01883104 0.0273735 ] | dL/dw2 = [0.08857237 0.09166328] | dL/db2 = [0.13718192]
Esperado (PyTorch): y = [0.7665197] 
  dL/dA1 =
 [[0.01883104 0.        ]
 [0.0273735  0.        ]] 
  dL/db1 = [0.01883104 0.0273735 ] | dL/dw2 = [0.08857237 0.09166329] | dL/db2 = [0.13718192]


## Conclusiones

En este laboratorio hemos explorado:

1. **Jacobiano y regla de la cadena**: el Jacobiano reúne en una matriz las derivadas parciales de una función vectorial, y el de una composición es el producto de los Jacobianos, $J_{f \circ g} = J_f(g)\, J_g$. Para una función escalar, ese producto da el gradiente como fila. Las diferencias finitas permiten estimar cualquier Jacobiano con una evaluación extra por variable, con una precisión de unos ocho dígitos que limita el equilibrio entre truncamiento y redondeo.
2. **Matriz Hessiana**: las derivadas segundas forman una matriz simétrica (teorema de Clairaut) que describe la curvatura de la función. Su estimación por diferencias centrales coincide con la analítica, y la simetría se observa también en la versión numérica.
3. **Backpropagation**: el gradiente de la pérdida respecto de los parámetros de una red es la regla de la cadena aplicada desde la pérdida hacia atrás. La derivada respecto de cada preactivación ($\partial L/\partial z$) se calcula una vez y se reutiliza: da el gradiente de los parámetros de esa capa ($\partial L/\partial w = \partial L/\partial z \cdot x$) y se propaga a la capa anterior a través de los pesos.
4. **Diferenciación automática**: PyTorch reproduce los gradientes manuales hasta la precisión de `float32`. Las diferencias finitas necesitan una evaluación de la red por parámetro, mientras que un *backward pass* obtiene todos los gradientes con un costo comparable al de una sola evaluación. Por eso el entrenamiento de redes con millones de parámetros usa diferenciación automática, y las diferencias finitas quedan como herramienta para verificar implementaciones.

## Declaración de uso de inteligencia artificial

> **Política del curso (syllabus).** Se permite usar herramientas de IA (ChatGPT, Copilot, Claude, etc.)
> *como apoyo para el aprendizaje*: entender conceptos, explorar ideas, depurar código o buscar
> explicaciones alternativas. **No** se permite usarlas para **resolver los ejercicios evaluados** ni
> para **verificar las respuestas antes de entregar**. Se espera que cada estudiante resuelva todos los
> problemas por sí mismo/a.

Completá esta declaración **escribiendo tu respuesta** donde aparece «…» (doble clic en esta celda para
editarla y luego Ctrl/Cmd + Enter para volver a renderizarla):

**1. ¿Usaste herramientas de IA en este laboratorio?** (Sí / No): Si

**2. ¿Cuál(es)?** (ChatGPT, Copilot, Claude, …; escribí "ninguna" si no usaste): Claude

**3. ¿Para qué la(s) usaste?** La use para ayudar a entender las ideas a grandes rasgos y bajarlas a tierra y sobre todo en los ejercicios de PyTorch

**4. Detalle breve:** «En qué ejercicios y de qué manera. Explicaciones en los ej 1 y 2 y aplicarlo en la parte 3

**Declaración de honestidad académica.** Declaro que resolví los ejercicios de este laboratorio por mí
mismo/a y que no utilicé herramientas de IA para resolver los ejercicios evaluados ni para verificar mis
respuestas antes de entregar, de acuerdo con la política del curso.

**Nombre y apellido:** Domingo Leiva          **Fecha:** 21/10/2026